In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# --------------------- Import VQNiche ---------------------
from vqniche.utils.parse_test_configs import *
from vqniche.initializers.initialize import *
from vqniche.utils.type_conversions import *
from vqniche.plotting import *

/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain

In [3]:
# --------------------- Import Libraries ---------------------
import os
import copy
import sys
import yaml
import pickle
from pathlib import Path
from dataclasses import dataclass

import scanpy as sc
import anndata as ad
import squidpy as sq

import numpy as np
import networkx as nx
import scipy.sparse as sp
from scipy.stats import pearsonr

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import pytorch_lightning as pl
import torch_geometric.transforms as T
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj
from torch_geometric.loader import DataLoader as BatchBuilder

# --------------------- Display Settings ---------------------
# display setting all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [111]:
save_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/paper/tables")
save_dir.mkdir(parents=True, exist_ok=True)

In [5]:
def collect_wandb_run_dirs(
        sweep_dir,
        type_of_run: Literal['run', 'offline-run']='run'
    ):
    wandb_run_dirs = []
    for d in os.listdir(sweep_dir / "wandb"):
        if type_of_run in d:
            # print(d)
            wandb_run_dirs.append(sweep_dir / "wandb" / d)
    return wandb_run_dirs

In [57]:
def build_sweep_test_results_df(
        wandb_run_dirs,
        split: Literal['1-test-patch', '3-test-patch']='1-test-patch'
    ):
    wandb_run_dir_dfs = []
    for wandb_run_dir in wandb_run_dirs:
        if isinstance(wandb_run_dir, str):
            cfg = f"{wandb_run_dir}/files/user_specified_config.yaml"
            test_res_file = f"{wandb_run_dir}/files/results/test_metrics.csv"
        else:
            cfg = wandb_run_dir/"files"/"user_specified_config.yaml"
            test_res_file = wandb_run_dir/"files"/"results"/"test_metrics.csv"
        with open(cfg, 'r') as f:
            cfg = yaml.safe_load(f)
        test_results_df = pd.read_csv(test_res_file)
        
        test_results_df['dataset'] = cfg['dataset']['dataset_name']
        test_results_df['seed'] = cfg['experiment']['seed']
        test_results_df['model'] = cfg['model']['model_name']
        test_results_df['split'] = split

        wandb_run_dir_dfs.append(test_results_df)
        
    sweep_results = pd.concat(
        objs=wandb_run_dir_dfs,
        ignore_index=True
    )

    return sweep_results

In [170]:
def build_clean_sweep_results_df(
        df: pd.DataFrame,
        metrics: List[str] = [
            "pearson_1hop_nbr",
            "pearson_gene_wise_1hop_nbr",
            "mmd_1hop_nbr",
            "mmd_pca_1hop_nbr",
        ]
    ):
    # --- Step 1: groupby and transpose ---
    df_long = df.melt(
        id_vars=["dataset", "seed", "model", "split", "epoch"], 
        var_name="metric", 
        value_name="value"
    )
    
    # --- Step 2: strip mode from metric name ---
    df_long["mode"] = df_long["metric"].str.extract(r"^(train|val|test)")
    df_long["metric"] = df_long["metric"].str.replace(r"^(train|val|test)_", "", regex=True)
    df_long = df_long[df_long['metric'].isin(metrics)]

    # --- Step 3: pretty names for metric ---
    metric_rename = {
        "pearson_1hop_nbr": "PC1",
        "pearson_gene_wise_1hop_nbr": "PG1",
        "pearson_cell_wise": "PC0",
        "mmd_1hop_nbr": "MMDC1",
        "mmd_pca_1hop_nbr": "MMD-PCA-C-1hop",
    }
    df_long["metric"] = df_long["metric"].map(metric_rename)

    # --- Step 4: pretty names for model ---
    model_rename = {
        "VQNiche": "SQUINT",
        "WFM": "WFM",
    }
    df_long["model"] = df_long["model"].map(model_rename)
    
    # --- Step 5: pretty names for dataset ---
    dataset_rename = {
        "xhs1000-39b_1p": "Skin (Human)",
        "mmb0-4b_1p": "Brain (Mouse)",
        "xhk1020-CV1-CV2-5b_1p": "Kidney (Human)",
    }
    df_long["dataset"] = df_long["dataset"].map(dataset_rename)
    
    # --- Step 6: pretty names for split ---
    split_rename = {
        "1-test-patch": "1 Patch",
        "3-test-patch": "3 Patches",
    }
    df_long["split"] = df_long["split"].map(split_rename)
    
    # --- Step 7: round to 4 decimals ---
    df_long['value'] = df_long['value'].round(4)
    
    df_long = df_long.drop_duplicates()
    
    return df_long

In [171]:
df_datasets = []
metrics = [
    "pearson_1hop_nbr",
    "pearson_cell_wise",
    # "pearson_gene_wise_1hop_nbr",
    "mmd_1hop_nbr",
    # "mmd_pca_1hop_nbr"
]

# Dataset: xhs1000-39b_1p-oriented-3

## 3 Test Regions

In [172]:
sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhs1000-39b_1p/sweep/VQNiche/batch=[2, 11, 12]/spatial_n_neighs_8/seed/20250917-210622")
wandb_run_dirs = collect_wandb_run_dirs(sweep_dir)
sweep_results = build_sweep_test_results_df(wandb_run_dirs, split='3-test-patch')
display(sweep_results.head(3))

df_long = build_clean_sweep_results_df(
                sweep_results,
                metrics,
            )
display(df_long.head(3))

df_datasets.append(df_long)

,test_codebook_utilization,test_pearson_cell_wise,test_pearson_1hop_nbr,test_pearson_gene_wise,test_pearson_gene_wise_1hop_nbr,test_mmd_1hop_nbr,test_mmd_pca_1hop_nbr,epoch,dataset,seed,model,split
0,0.0250,0.410830,0.738857,0.105257,0.295920,0.107091,0.004112,19,xhs1000-39b_1p,0,VQNiche,3-test-patch
1,0.0196,0.379068,0.666443,0.083465,0.245880,0.114131,0.003866,19,xhs1000-39b_1p,1,VQNiche,3-test-patch
2,0.0208,0.389794,0.683567,0.092848,0.259767,0.113180,0.004081,18,xhs1000-39b_1p,2,VQNiche,3-test-patch


,dataset,seed,model,split,epoch,metric,value,mode
4,Skin (Human),0,SQUINT,3 Patches,19,PC0,0.4108,test
5,Skin (Human),1,SQUINT,3 Patches,19,PC0,0.3791,test
6,Skin (Human),2,SQUINT,3 Patches,18,PC0,0.3898,test


# Dataset: mmb04-4b_1p

## 3 Test Regions

In [173]:
sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/mmb0-4b_1p/sweep/VQNiche/batch=[0, 1, 2, 3]/spatial_n_neighs_8/seed/20250917-210552")
wandb_run_dirs = collect_wandb_run_dirs(sweep_dir)
sweep_results = build_sweep_test_results_df(wandb_run_dirs, split='3-test-patch')
display(sweep_results.head(3))

df_long = build_clean_sweep_results_df(
                sweep_results,
                metrics,
            )
display(df_long.head(3))

df_datasets.append(df_long)

,test_codebook_utilization,test_pearson_cell_wise,test_pearson_1hop_nbr,test_pearson_gene_wise,test_pearson_gene_wise_1hop_nbr,test_mmd_1hop_nbr,test_mmd_pca_1hop_nbr,epoch,dataset,seed,model,split
0,0.0264,0.636755,0.893444,0.110635,0.361938,0.047834,0.002695,19,mmb0-4b_1p,0,VQNiche,3-test-patch
1,0.0254,0.637673,0.891722,0.105236,0.350348,0.050114,0.002722,19,mmb0-4b_1p,1,VQNiche,3-test-patch
2,0.0236,0.537588,0.802825,0.095811,0.305757,0.077985,0.002611,19,mmb0-4b_1p,2,VQNiche,3-test-patch


,dataset,seed,model,split,epoch,metric,value,mode
4,Brain (Mouse),0,SQUINT,3 Patches,19,PC0,0.6368,test
5,Brain (Mouse),1,SQUINT,3 Patches,19,PC0,0.6377,test
6,Brain (Mouse),2,SQUINT,3 Patches,19,PC0,0.5376,test


# Dataset: xhk1020-CV1-CV2-5b_1p

## 3 Test Regions

In [174]:
sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhk1020-CV1-CV2-5b_1p/standalone/VQNiche/batch=[0, 1, 2, 3, 4]/spatial_n_neighs_8")
wandb_run_dirs = collect_wandb_run_dirs(sweep_dir)
sweep_results = build_sweep_test_results_df(wandb_run_dirs, split='3-test-patch')
display(sweep_results.head(3))

df_long = build_clean_sweep_results_df(
                sweep_results,
                metrics,
            )
display(df_long.head(3))

df_datasets.append(df_long)

,test_codebook_utilization,test_pearson_cell_wise,test_pearson_1hop_nbr,test_pearson_gene_wise,test_pearson_gene_wise_1hop_nbr,test_mmd_1hop_nbr,test_mmd_pca_1hop_nbr,epoch,dataset,seed,model,split
0,0.0218,0.480199,0.792819,0.052518,0.194196,0.043979,0.000841,0,xhk1020-CV1-CV2-5b_1p,0,VQNiche,3-test-patch
1,0.0218,0.480199,0.792819,0.052518,0.194196,0.043979,0.000841,0,xhk1020-CV1-CV2-5b_1p,0,VQNiche,3-test-patch


,dataset,seed,model,split,epoch,metric,value,mode
2,Kidney (Human),0,SQUINT,3 Patches,0,PC0,0.4802,test
4,Kidney (Human),0,SQUINT,3 Patches,0,PC1,0.7928,test
10,Kidney (Human),0,SQUINT,3 Patches,0,MMDC1,0.0440,test


# Final Table

In [175]:
df_avg = pd.concat(
            objs=df_datasets,
            ignore_index=True
        )
df_avg = df_avg.groupby(
        ['dataset', 'split', 'model', 'metric']
        )['value'].mean().reset_index()
df_avg = df_avg.round(4)
display(df_avg)

,dataset,split,model,metric,value
0,Brain (Mouse),3 Patches,SQUINT,MMDC1,0.0586
1,Brain (Mouse),3 Patches,SQUINT,PC0,0.6040
2,Brain (Mouse),3 Patches,SQUINT,PC1,0.8626
3,Kidney (Human),3 Patches,SQUINT,MMDC1,0.0440
4,Kidney (Human),3 Patches,SQUINT,PC0,0.4802
5,Kidney (Human),3 Patches,SQUINT,PC1,0.7928
6,Skin (Human),3 Patches,SQUINT,MMDC1,0.1115
7,Skin (Human),3 Patches,SQUINT,PC0,0.3932
8,Skin (Human),3 Patches,SQUINT,PC1,0.6963


In [176]:
df_pivot = df_avg.pivot(index='model', columns=['dataset','metric'], values='value')
model_order = ['WFM', 'SQUINT']
df_pivot = df_pivot.reindex(model_order).reset_index()
df_pivot = df_pivot.fillna('--')
for i in range(df_pivot.shape[1]-1):
    df_pivot.insert(i*2+1,f'amps-{i}','&')
i += 1
df_pivot.insert(i*2+1,f'newline-{i}','\\\\')
display(df_pivot)

dataset,model,amps-0,Brain (Mouse),amps-1,Brain (Mouse),amps-2,Brain (Mouse),amps-3,Kidney (Human),amps-4,Kidney (Human),amps-5,Kidney (Human),amps-6,Skin (Human),amps-7,Skin (Human),amps-8,Skin (Human),newline-9
metric,,,MMDC1,,PC0,,PC1,,MMDC1,,PC0,,PC1,,MMDC1,,PC0,,PC1,
0,WFM,&,--,&,--,&,--,&,--,&,--,&,--,&,--,&,--,&,--,\\
1,SQUINT,&,0.0586,&,0.604,&,0.8626,&,0.044,&,0.4802,&,0.7928,&,0.1115,&,0.3932,&,0.6963,\\


In [178]:
top_str = """\\begin{table*}[t]\n
\\captionsetup[sub]{skip=0pt} \n
\\centering  \n
\\setlength\\tabcolsep{3pt} \n
\\caption{2D Imputation Results.}
\label{tab:2D_imputation_results} \n
"""

header_str = """\\begin{tabular}{lcccccccc} \n
\\toprule \n
\\multirow{2}[2]{*}{\\textbf{Model}} & \\multicolumn{4}{c}{\\textbf{Brain (Mouse)}} & \\multicolumn{4}{c}{\\textbf{Skin (Human)}} \\\\ \n
\\cmidrule(lr){2-5} \\cmidrule(lr){6-9} \n
& MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC1 $\\uparrow$ & PG1 $\\uparrow$ & MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC1 $\\uparrow$ & PG1 $\\uparrow$ \\\\ \n
\\midrule \n
"""

header_str = """\\begin{tabular}{lcccccccc} \n
\\toprule \n
\\multirow{2}[2]{*}{\\textbf{Model}} & \\multicolumn{4}{c}{\\textbf{Brain (Mouse)}} & \\multicolumn{4}{c}{\\textbf{Skin (Human)}} \\\\ \n
\\cmidrule(lr){2-5} \\cmidrule(lr){6-9} \n
& MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ & MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ \\\\ \n
\\midrule \n
"""
   # WFM &      0.0542 &      0.1314 &     -- &      -- &     0.0887 &      0.2026 &      -- &      -- \\

header_str = """\\begin{tabular}{lcccccccccccc} \n
\\toprule \n
\\multirow{2}[2]{*}{\\textbf{Model}} & \\multicolumn{4}{c}{\\textbf{Brain (Mouse)}} & \\multicolumn{4}{c}{\\textbf{Skin (Human)}} & \\multicolumn{4}{c}{\\textbf{Skin (Kidney)}} \\\\ \n
\\cmidrule(lr){2-5} \\cmidrule(lr){6-9} \\cmidrule(lr){10-13} \n
& MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ & MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ & MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ \\\\ \n
\\midrule \n
"""

   # WFM &      0.0542 &      0.1314 &     -- &      -- &     0.0887 &      0.2026 &      -- &      --    &      0.0039 &      0.1118 &     -- &      -- \\

header_str = """\\begin{tabular}{lccccccccc} \n
\\toprule \n
\\multirow{2}[2]{*}{\\textbf{Model}} & \\multicolumn{3}{c}{\\textbf{Brain (Mouse)}} & \\multicolumn{3}{c}{\\textbf{Kidney (Human)}} & \\multicolumn{3}{c}{\\textbf{Skin (Human)}}  \\\\ \n
\\cmidrule(lr){2-4} \\cmidrule(lr){5-7} \\cmidrule(lr){8-10} \n
& MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ \\\\ \n
\\midrule \n
"""

   # WFM &      0.1314 &     -- &      -- &  0.1118 &     -- &      -- &  0.2026 &      -- &      --         \\

data_str = df_pivot.to_string(header=False, index=False)

footer_str =  "\n\\bottomrule \n \\end{tabular} \n"

bottom_str = '\\end{table*}'


fname = save_dir / "2D_imputation_results.tex"
with open(fname, 'w') as fp:
    fp.write(top_str)
    fp.write(header_str)
    fp.write(data_str)
    fp.write(footer_str)
    fp.write(bottom_str)